# Parte 1 — Enriquecimiento de los avisos

Primero unimos las ventas de `secondary_market.csv` con los precios mensuales por distrito y con los datos de estaciones de metro. Usamos `LEFT JOIN` porque cada aviso es una observación que queremos conservar. El promedio distrital corresponde al mismo mes del aviso; al pensar en una predicción real debemos comprobar si ese dato ya estaba disponible en la fecha de publicación.

## 1. Carga y clave temporal

Leemos los tres archivos de `../data_sets/` y obtenemos `year_month` a partir de `date_posted` para relacionar cada aviso con su distrito y mes. Convertimos las fechas antes del cruce para detectar formatos incorrectos.

Iniciamos el servidor local de MLflow indicado en `README.md` antes de ejecutar el notebook. Al terminar guardamos en un run los conteos de limpieza y la ruta del CSV final.

In [17]:
from pathlib import Path

import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.statsmodels

mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('UTN_Mercado_Secundario')

DATA_DIR = Path('..') / 'data_sets'
ventas = pd.read_csv(DATA_DIR / 'secondary_market.csv')
precios_distrito = pd.read_csv(DATA_DIR / 'district_prices_monthly.csv')
estaciones_metro = pd.read_csv(DATA_DIR / 'metro_stations.csv')

ventas['year_month'] = pd.to_datetime(ventas['date_posted'], errors='raise').dt.strftime('%Y-%m')
print(f'Ventas originales: {len(ventas):,} filas')
print(f'Precios mensuales: {len(precios_distrito):,} filas')
print(f'Estaciones de metro: {len(estaciones_metro):,} filas')

Ventas originales: 50,000 filas
Precios mensuales: 9,804 filas
Estaciones de metro: 104 filas


## 2. Precios del distrito en el mes del aviso

Unimos por `district` y `year_month` el precio promedio por metro cuadrado, su variación mensual y la tasa del Banco Central. Comprobamos que cada distrito y mes tenga una sola fila para no duplicar propiedades. Al principio de algunas series no hay variación respecto de un mes anterior, por lo que conservamos esos valores nulos hasta la limpieza.

In [18]:
columnas_mensuales = [
    'district', 'year_month', 'secondary_price_per_sqm',
    'secondary_mom_change_pct', 'cbr_key_rate_pct',
]
ventas_con_distrito = ventas.merge(
    precios_distrito[columnas_mensuales],
    on=['district', 'year_month'],
    how='left',
    validate='many_to_one',
    indicator='_join_distrito',
)
assert len(ventas_con_distrito) == len(ventas), 'El join mensual cambió la cantidad de avisos'
assert ventas_con_distrito['id'].equals(ventas['id']), 'El join mensual alteró el orden de los avisos'
print(ventas_con_distrito['_join_distrito'].value_counts(dropna=False))

_join_distrito
both          50000
left_only         0
right_only        0
Name: count, dtype: int64


## 3. Datos de la estación de metro

Relacionamos `metro_station` con `station_name` para sumar `year_opened` y la distancia de la estación al centro. Ya teníamos `to_center_km` para el aviso, por eso guardamos la nueva medida como `metro_station_to_center_km`: se refieren a lugares diferentes.

In [19]:
metro_para_join = estaciones_metro[
    ['station_name', 'year_opened', 'to_center_km']
].rename(columns={'to_center_km': 'metro_station_to_center_km'})
ventas_enriquecidas = ventas_con_distrito.merge(
    metro_para_join,
    left_on='metro_station',
    right_on='station_name',
    how='left',
    validate='many_to_one',
    indicator='_join_metro',
)
assert len(ventas_enriquecidas) == len(ventas), 'El join de metro cambió la cantidad de avisos'
assert ventas_enriquecidas['id'].equals(ventas['id']), 'El join de metro alteró el orden de los avisos'
print(ventas_enriquecidas['_join_metro'].value_counts(dropna=False))

_join_metro
both          50000
left_only         0
right_only        0
Name: count, dtype: int64


## 4. Revisión de los cruces

Contamos cuántos avisos encontraron información en cada tabla y verificamos que los 50.000 originales sigan presentes y en el mismo orden. Las columnas `_join_distrito` y `_join_metro` nos permiten revisar esas coincidencias.

In [20]:
columnas_agregadas = [
    'secondary_price_per_sqm', 'secondary_mom_change_pct', 'cbr_key_rate_pct',
    'year_opened', 'metro_station_to_center_km',
]
print(f'Filas antes/después: {len(ventas):,} / {len(ventas_enriquecidas):,}')
print('Sin coincidencia por distrito y mes:', (ventas_enriquecidas['_join_distrito'] == 'left_only').sum())
print('Sin coincidencia por estación:', (ventas_enriquecidas['_join_metro'] == 'left_only').sum())
print('Faltantes en columnas agregadas:')
print(ventas_enriquecidas[columnas_agregadas].isna().sum().to_string())
display(ventas_enriquecidas[[
    'id', 'date_posted', 'district', 'year_month', 'metro_station',
    *columnas_agregadas, '_join_distrito', '_join_metro',
]].head())

Filas antes/después: 50,000 / 50,000
Sin coincidencia por distrito y mes: 0
Sin coincidencia por estación: 0
Faltantes en columnas agregadas:
secondary_price_per_sqm         0
secondary_mom_change_pct      678
cbr_key_rate_pct                0
year_opened                     0
metro_station_to_center_km      0


,id,date_posted,district,year_month,metro_station,secondary_price_per_sqm,secondary_mom_change_pct,cbr_key_rate_pct,year_opened,metro_station_to_center_km,_join_distrito,_join_metro
0,S000001,2024-11-23,Begovoy,2024-11,Sokol,393000,3.98,21.0,1938,8.72,both,both
1,S000002,2025-08-13,Sokolniki,2025-08,Sokolniki,355200,3.57,17.0,1935,5.45,both,both
2,S000003,2022-05-06,Sosenskoye,2022-05,Polyanka,192400,0.34,9.5,1986,1.80,both,both
3,S000004,2025-04-08,Sviblovo,2025-04,Sviblovo,270900,-2.67,21.0,1978,11.34,both,both
4,S000005,2023-07-16,Begovoy,2023-07,Sapyornoye Pole,367300,0.94,13.0,2018,5.98,both,both


## 5. Dataset enriquecido

Guardamos el resultado de los dos cruces con las columnas originales y los datos agregados. Así podemos volver a este punto si necesitamos revisar algún registro.

In [21]:
RUTA_SALIDA = Path('secondary_market_enriquecido.csv')
ventas_enriquecidas.to_csv(RUTA_SALIDA, index=False)
ventas_guardadas = pd.read_csv(RUTA_SALIDA, usecols=['id'])
assert ventas_guardadas['id'].equals(ventas['id']), 'El CSV no conserva los avisos originales en orden'
print(f'Dataset guardado en: {RUTA_SALIDA.resolve()}')
print(f'Filas guardadas: {len(ventas_guardadas):,}')

Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\2round\secondary_market_enriquecido.csv
Filas guardadas: 50,000


# Parte 2 — Limpieza e ingeniería de variables

## 2.1. Coherencia de superficies

Eliminamos los 247 avisos en los que `living_area + kitchen_area > total_area`: la suma de los espacios interiores no puede superar el área total. La comparación es estricta, por lo que conservamos los casos donde la suma es igual al total. Quedan 49.753 avisos.

In [22]:
ventas_parte_1 = pd.read_csv(Path('secondary_market_enriquecido.csv'))
areas_inconsistentes = (
    ventas_parte_1['living_area'] + ventas_parte_1['kitchen_area']
    > ventas_parte_1['total_area']
)
ventas_areas_validas = ventas_parte_1.loc[~areas_inconsistentes].copy()
assert len(ventas_areas_validas) + int(areas_inconsistentes.sum()) == len(ventas_parte_1)
assert not ((ventas_areas_validas['living_area'] + ventas_areas_validas['kitchen_area']) > ventas_areas_validas['total_area']).any()
print(f'Filas originales: {len(ventas_parte_1):,}')
print(f'Filas eliminadas por superficies inconsistentes: {areas_inconsistentes.sum():,}')
print(f'Filas conservadas: {len(ventas_areas_validas):,}')

Filas originales: 50,000
Filas eliminadas por superficies inconsistentes: 247
Filas conservadas: 49,753


### Dataset con superficies coherentes

Guardamos las 49.753 filas resultantes para poder rastrear qué cambió en esta primera regla de limpieza.

In [23]:
RUTA_AREAS_VALIDAS = Path('secondary_market_enriquecido_areas_validas.csv')
ventas_areas_validas.to_csv(RUTA_AREAS_VALIDAS, index=False)
ids_guardados = pd.read_csv(RUTA_AREAS_VALIDAS, usecols=['id'])['id']
assert ids_guardados.equals(ventas_areas_validas['id'].reset_index(drop=True))
print(f'Dataset guardado en: {RUTA_AREAS_VALIDAS.resolve()}')
print(f'Filas guardadas: {len(ids_guardados):,}')

Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\2round\secondary_market_enriquecido_areas_validas.csv
Filas guardadas: 49,753


## 2.2. Coherencia entre construcción y publicación

Calculamos `posted_year` a partir de `date_posted` y eliminamos los 1.266 registros con `building_year > posted_year`. En un aviso del mercado secundario, el edificio no debería figurar como construido después de la publicación. Quedan 48.487 propiedades; preferimos descartar esas inconsistencias temporales antes que inventar un año de construcción.

In [24]:
ventas_con_anio = ventas_areas_validas.copy()
ventas_con_anio['posted_year'] = pd.to_datetime(
    ventas_con_anio['date_posted'], errors='raise'
).dt.year
construccion_futura = ventas_con_anio['building_year'] > ventas_con_anio['posted_year']
ventas_areas_y_anios_validos = ventas_con_anio.loc[~construccion_futura].copy()
assert len(ventas_areas_y_anios_validos) + int(construccion_futura.sum()) == len(ventas_con_anio)
assert not (ventas_areas_y_anios_validos['building_year'] > ventas_areas_y_anios_validos['posted_year']).any()
print(f'Filas después del control de superficies: {len(ventas_con_anio):,}')
print(f'Filas eliminadas por construcción posterior al aviso: {construccion_futura.sum():,}')
print(f'Filas conservadas después de ambos controles: {len(ventas_areas_y_anios_validos):,}')

Filas después del control de superficies: 49,753
Filas eliminadas por construcción posterior al aviso: 1,266
Filas conservadas después de ambos controles: 48,487


### Dataset con superficies y años coherentes

Guardamos las 48.487 filas que pasaron los dos controles físicos y temporales.

In [25]:
RUTA_AREAS_Y_ANIOS_VALIDOS = Path('secondary_market_enriquecido_areas_y_anios_validos.csv')
ventas_areas_y_anios_validos.to_csv(RUTA_AREAS_Y_ANIOS_VALIDOS, index=False)
ids_guardados = pd.read_csv(RUTA_AREAS_Y_ANIOS_VALIDOS, usecols=['id'])['id']
assert ids_guardados.equals(ventas_areas_y_anios_validos['id'].reset_index(drop=True))
print(f'Dataset guardado en: {RUTA_AREAS_Y_ANIOS_VALIDOS.resolve()}')
print(f'Filas guardadas: {len(ids_guardados):,}')

Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\2round\secondary_market_enriquecido_areas_y_anios_validos.csv
Filas guardadas: 48,487


## 2.3. Datos faltantes en la variación mensual

Encontramos 622 nulos en `secondary_mom_change_pct` después de la limpieza y los completamos con cero para poder usar la columna en los cálculos. Ese cero es una convención de trabajo: no implica que hayamos observado una variación real igual a cero.

In [26]:
ventas_parte_2 = ventas_areas_y_anios_validos.copy()
nulos_mom_antes = int(ventas_parte_2['secondary_mom_change_pct'].isna().sum())
ventas_parte_2['secondary_mom_change_pct'] = ventas_parte_2['secondary_mom_change_pct'].fillna(0)
assert len(ventas_parte_2) == len(ventas_areas_y_anios_validos)
assert ventas_parte_2['secondary_mom_change_pct'].notna().all()
print(f'Valores nulos reemplazados con 0: {nulos_mom_antes:,}')
print(f'Filas conservadas: {len(ventas_parte_2):,}')

Valores nulos reemplazados con 0: 622
Filas conservadas: 48,487


### Dataset con la variación mensual completa

Conservamos las 48.487 filas y guardamos esta versión para identificar la imputación realizada.

In [27]:
RUTA_PARTE_2 = Path('secondary_market_enriquecido_areas_anios_mom_0.csv')
ventas_parte_2.to_csv(RUTA_PARTE_2, index=False)
ventas_parte_2_guardadas = pd.read_csv(RUTA_PARTE_2)
assert len(ventas_parte_2_guardadas) == len(ventas_parte_2)
assert ventas_parte_2_guardadas['id'].equals(ventas_parte_2['id'].reset_index(drop=True))
assert ventas_parte_2_guardadas['secondary_mom_change_pct'].notna().all()
print(f'Dataset guardado en: {RUTA_PARTE_2.resolve()}')
print(f'Filas guardadas: {len(ventas_parte_2_guardadas):,}')

Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\2round\secondary_market_enriquecido_areas_anios_mom_0.csv
Filas guardadas: 48,487


## 2.4. Columnas que retiramos

Eliminamos `id`, `date_posted`, `year_month`, `price_rub` y `station_name`. En particular, `price_rub` junto con `total_area` permite reconstruir el objetivo `price_per_sqm` mediante una división, por lo que no puede entrar como predictor. `station_name` repite la estación indicada en `metro_station`; mantenemos `metro_line`, que es otra variable.

In [28]:
columnas_a_eliminar = ['id', 'date_posted', 'year_month', 'price_rub', 'station_name']
columnas_ausentes = ['line', 'lat_metro', 'lon_metro']
assert set(columnas_a_eliminar).issubset(ventas_parte_2.columns), 'Faltan columnas esperadas para eliminar'
assert not set(columnas_ausentes).intersection(ventas_parte_2.columns), 'Revisar columnas de metro antes de eliminarlas'
ventas_parte_2_columnas = ventas_parte_2.drop(columns=columnas_a_eliminar).copy()
assert len(ventas_parte_2_columnas) == len(ventas_parte_2)
assert 'price_per_sqm' in ventas_parte_2_columnas.columns
assert len(ventas_parte_2_columnas.columns) == len(ventas_parte_2.columns) - len(columnas_a_eliminar)
print('Columnas eliminadas:', columnas_a_eliminar)
print('Columnas ausentes:', columnas_ausentes)
print(f'Filas conservadas: {len(ventas_parte_2_columnas):,}')
print(f'Columnas restantes: {len(ventas_parte_2_columnas.columns)}')

Columnas eliminadas: ['id', 'date_posted', 'year_month', 'price_rub', 'station_name']
Columnas ausentes: ['line', 'lat_metro', 'lon_metro']
Filas conservadas: 48,487
Columnas restantes: 31


### Dataset sin identificadores ni precio total

Guardamos la versión con 31 columnas y el objetivo `price_per_sqm` intacto.

In [29]:
RUTA_PARTE_2_COLUMNAS = Path('secondary_market_parte_2_sin_columnas_descartadas.csv')
ventas_parte_2_columnas.to_csv(RUTA_PARTE_2_COLUMNAS, index=False)
ventas_parte_2_columnas_guardadas = pd.read_csv(RUTA_PARTE_2_COLUMNAS)
assert len(ventas_parte_2_columnas_guardadas) == len(ventas_parte_2)
assert ventas_parte_2_columnas_guardadas.columns.equals(ventas_parte_2_columnas.columns)
assert ventas_parte_2_columnas_guardadas['price_per_sqm'].equals(ventas_parte_2['price_per_sqm'].reset_index(drop=True))
print(f'Dataset guardado en: {RUTA_PARTE_2_COLUMNAS.resolve()}')
print(f'Filas: {len(ventas_parte_2_columnas_guardadas):,}; columnas: {len(ventas_parte_2_columnas_guardadas.columns)}')

Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\2round\secondary_market_parte_2_sin_columnas_descartadas.csv
Filas: 48,487; columnas: 31


## 2.5. Antigüedad del edificio y piso relativo

Calculamos `antiguedad = posted_year - building_year` y `floor_ratio = floor / total_floors`. Así distinguimos la edad del edificio al publicarse el aviso y la posición del departamento dentro de edificios de distintas alturas. Revisamos que `total_floors` sea mayor que cero antes de dividir.

In [30]:
ventas_parte_2_derivadas = ventas_parte_2_columnas.copy()
columnas_necesarias = ['posted_year', 'building_year', 'floor', 'total_floors']
assert ventas_parte_2_derivadas[columnas_necesarias].notna().all().all(), 'Hay superficies, pisos o años faltantes'
assert (ventas_parte_2_derivadas['total_floors'] > 0).all(), 'total_floors debe ser positivo'
ventas_parte_2_derivadas['antiguedad'] = (
    ventas_parte_2_derivadas['posted_year'] - ventas_parte_2_derivadas['building_year']
)
ventas_parte_2_derivadas['floor_ratio'] = (
    ventas_parte_2_derivadas['floor'] / ventas_parte_2_derivadas['total_floors']
)
assert (ventas_parte_2_derivadas['antiguedad'] >= 0).all()
assert len(ventas_parte_2_derivadas) == len(ventas_parte_2_columnas)
print(f'Filas conservadas: {len(ventas_parte_2_derivadas):,}')
print(ventas_parte_2_derivadas[['antiguedad', 'floor_ratio']].describe().to_string())

Filas conservadas: 48,487
         antiguedad   floor_ratio
count  48487.000000  48487.000000
mean      35.421742      0.545252
std       23.209538      0.289117
min        0.000000      0.040000
25%       14.000000      0.294118
50%       35.000000      0.555556
75%       55.000000      0.800000
max       91.000000      1.000000


### Dataset para continuar el EDA

Guardamos las 48.487 observaciones y 33 columnas resultantes. Este archivo es el punto de partida de la sección 3.

In [31]:
RUTA_PARTE_2_DERIVADAS = Path('secondary_market_parte_2_con_antiguedad_y_floor_ratio.csv')
ventas_parte_2_derivadas.to_csv(RUTA_PARTE_2_DERIVADAS, index=False)
ventas_derivadas_guardadas = pd.read_csv(RUTA_PARTE_2_DERIVADAS)
assert len(ventas_derivadas_guardadas) == len(ventas_parte_2_derivadas)
assert {'antiguedad', 'floor_ratio'}.issubset(ventas_derivadas_guardadas.columns)
assert ventas_derivadas_guardadas['price_per_sqm'].equals(ventas_parte_2_columnas['price_per_sqm'].reset_index(drop=True))
print(f'Dataset guardado en: {RUTA_PARTE_2_DERIVADAS.resolve()}')
print(f'Filas: {len(ventas_derivadas_guardadas):,}; columnas: {len(ventas_derivadas_guardadas.columns)}')

with mlflow.start_run(run_name='Data_Prep_y_Limpieza'):
    mlflow.log_params({
        'filas_originales': len(ventas),
        'filas_areas_inconsistentes_eliminadas': int(areas_inconsistentes.sum()),
        'filas_construccion_futura_eliminadas': int(construccion_futura.sum()),
        'nulos_variacion_mensual_completados': nulos_mom_antes,
        'filas_finales': len(ventas_derivadas_guardadas),
        'csv_generado': RUTA_PARTE_2_DERIVADAS.name,
        'csv_limpio': str(RUTA_PARTE_2_DERIVADAS.resolve()),
    })
    mlflow.log_artifact(str(RUTA_PARTE_2_DERIVADAS), artifact_path='dataset')
    mlflow.log_dict({
        'archivos_fuente': ['secondary_market.csv', 'district_prices_monthly.csv', 'metro_stations.csv'],
        'filas_iniciales': len(ventas),
        'filas_finales': len(ventas_derivadas_guardadas),
        'areas_inconsistentes_descartadas': int(areas_inconsistentes.sum()),
        'construccion_futura_descartada': int(construccion_futura.sum()),
        'nulos_variacion_mensual_completados': nulos_mom_antes,
    }, 'diagnostico/resumen_limpieza.json')

Dataset guardado en: C:\Users\Rodrigo\Documents\Facultad\Analisis de datos\Parcial\2round\secondary_market_parte_2_con_antiguedad_y_floor_ratio.csv
Filas: 48,487; columnas: 33
🏃 View run Data_Prep_y_Limpieza at: http://127.0.0.1:5000/#/experiments/5/runs/32e616ee70b74ed19d46ebb57c6cf465
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
